# Discrete non-text sequences

Synthetic categorical trajectories. The transition table includes an explicit END outcome. This is not a molecular MSM or a kinetic estimator.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import markovify
from markovify.nlp import Corpus
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUT = ROOT / "notebooks" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
print(markovify.__file__)

In [ ]:
from markovify.chain import BEGIN, END
import random
rng = random.Random(SEED)
runs = []
for _ in range(100):
    run = ["closed"]
    for _ in range(19):
        run.append(rng.choices(["closed","open"],weights=[8,2] if run[-1]=="closed" else [3,7])[0])
    runs.append(run)
chain = markovify.Chain(runs,state_size=1,random_state=SEED)
counts = pd.DataFrame.from_dict({s[0]:v for s,v in chain.model.items()},orient="index").fillna(0)
prob = counts.div(counts.sum(axis=1),axis=0)
display(prob)
fig, ax = plt.subplots()
im = ax.imshow(prob.values,vmin=0,vmax=1)
ax.set_xticks(range(len(prob.columns)),prob.columns,rotation=45)
ax.set_yticks(range(len(prob.index)),prob.index)
ax.set_xlabel("Next token")
ax.set_ylabel("Current state")
fig.colorbar(im,label="Probability")
plt.tight_layout()
plt.show()

In [ ]:
simulated = [chain.walk() for _ in range(200)]
pd.Series([len(r) for r in simulated]).plot.hist(bins=20,title="Simulated sequence lengths")
plt.xlabel("Steps")
plt.show()
chain.save(OUT / "discrete_chain.json.gz")
assert markovify.load_model(OUT / "discrete_chain.json.gz").model == chain.model